# Deep Learning (Foundation Models) — Proof of Concept

This notebook runs **expanding-window backtests** on hourly Czech gas consumption
(2013–2025) using **pretrained foundation models**:

- `amazon/chronos-2` (Chronos-2)
- `time-series-foundation-models/Lag-Llama`
- `Salesforce/moirai-1.0-R-base`

Workflow:
1) **Zero-shot** inference with author weights
2) **Training / fine-tuning** on your data (minimal POC)

Outputs:
- No plots. Only **pandas DataFrames**.
- Metrics: **SMAPE**, **MAE**, **MSE**, **R²**.
- Special focus: **Russia–Ukraine conflict** (`2022-02-24`) via pre/post slices.


In [57]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Callable, Iterable

import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

DATA_PATH = (
    Path('..')
    / 'data'
    / 'processed'
    / 'merged'
    / 'merged_all_years.csv'
).resolve()

TARGET_COL = 'consumption_total'

PREDICTION_LENGTH = 24  # hours ahead
WINDOW_STRIDE = 24  # evaluate 1x per day; set to 1 for full hourly rolling

MAX_CONTEXT_LENGTH = 2048  # cap for speed + model limits

CONFLICT_DATE = pd.Timestamp('2022-02-24 00:00:00')
TZ = None  # data is already in local calendar time; keep naive timestamps

SEED = 42
np.random.seed(SEED)

SHOW_PROGRESS = True

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 200)


In [58]:
def build_timestamp(df: pd.DataFrame) -> pd.Series:
    ts = pd.to_datetime(df[['year', 'month', 'day', 'hour']], errors='coerce')
    if TZ is not None:
        ts = ts.dt.tz_localize(TZ)
    return ts


def dedupe_by_timestamp(df: pd.DataFrame, timestamp_col: str) -> pd.DataFrame:
    if df[timestamp_col].isna().any():
        raise ValueError('Found NaT timestamps; fix parsing before de-duping.')

    dup_count = int(df.duplicated(timestamp_col).sum())
    if dup_count == 0:
        return df

    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

    agg: dict[str, str] = {c: 'mean' for c in numeric_cols}
    for c in ['holiday', 'before_holiday', 'day_of_week', 'weather_code']:
        if c in agg:
            agg[c] = 'max'

    out = (
        df.groupby(timestamp_col, as_index=False)
        .agg(agg)
        .sort_values(timestamp_col, kind='stable')
        .reset_index(drop=True)
    )
    return out


def interpolate_small_gaps(s: pd.Series, limit: int = 6) -> pd.Series:
    out = s.copy()
    out = out.interpolate(method='linear', limit=limit, limit_direction='both')
    out = out.ffill().bfill()
    return out


df_raw = pd.read_csv(DATA_PATH)
df_raw['timestamp'] = build_timestamp(df_raw)

df = (
    df_raw.drop(columns=['year', 'month', 'day', 'hour'])
    .sort_values('timestamp', kind='stable')
    .reset_index(drop=True)
)

df = dedupe_by_timestamp(df, timestamp_col='timestamp')

missing_before = int(df[TARGET_COL].isna().sum())
df[TARGET_COL] = interpolate_small_gaps(df[TARGET_COL], limit=6)
missing_after = int(df[TARGET_COL].isna().sum())

print('rows:', len(df))
print('timestamp min/max:', df['timestamp'].min(), df['timestamp'].max())
print('target missing before/after:', missing_before, missing_after)


rows: 113952
timestamp min/max: 2013-01-01 00:00:00 2025-12-31 23:00:00
target missing before/after: 32 0


In [59]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.maximum(denom, 1e-8)
    return float(200.0 * np.mean(np.abs(y_pred - y_true) / denom))


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    return {
        'smape': smape(y_true, y_pred),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'mse': float(mean_squared_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
    }


def year_bounds(year: int) -> tuple[pd.Timestamp, pd.Timestamp]:
    start = pd.Timestamp(datetime(year, 1, 1))
    end = pd.Timestamp(datetime(year + 1, 1, 1))
    return start, end


def iter_test_origins(test_df: pd.DataFrame) -> Iterable[int]:
    max_i = len(test_df) - PREDICTION_LENGTH
    i = 0
    while i <= max_i:
        yield i
        i += WINDOW_STRIDE


In [60]:
import torch
from gluonts.dataset.common import ListDataset
from gluonts.torch.distributions.studentT import StudentTOutput
from gluonts.torch.modules.loss import NegativeLogLikelihood
from lag_llama.gluon.estimator import LagLlamaEstimator
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule


@dataclass(frozen=True)
class ForecastResult:
    y_pred: np.ndarray


class BaseForecaster:
    name: str
    mode: str

    def forecast(self, context: np.ndarray) -> ForecastResult:
        raise NotImplementedError


class Chronos2Forecaster(BaseForecaster):
    def __init__(
        self,
        repo_id: str = 'amazon/chronos-2',
        device: str = 'auto',
        num_samples: int = 21,
    ) -> None:
        from chronos import Chronos2Pipeline

        if device == 'auto':
            device = 'cuda' if torch.cuda.is_available() else 'cpu'

        self.name = 'chronos-2'
        self.mode = 'pretrained'
        self._pipe = Chronos2Pipeline.from_pretrained(repo_id)
        self._pipe.model.to(device)  # type: ignore
        self._num_samples = num_samples

    def forecast(self, context: np.ndarray) -> ForecastResult:
        x = np.asarray(context, dtype=np.float32)
        x = x[-MAX_CONTEXT_LENGTH:]
        x3 = x[None, None, :]

        with torch.no_grad():
            out = self._pipe.predict(
                x3,
                prediction_length=PREDICTION_LENGTH,
                batch_size=1,
                context_length=min(len(x), MAX_CONTEXT_LENGTH),
            )

        samples = out[0][0]
        y_pred = samples.float().mean(dim=0).cpu().numpy()
        return ForecastResult(y_pred=y_pred)


def _patch_lag_llama_lags_handling() -> None:
    import lag_llama.gluon.estimator as ll_est

    orig_fn = ll_est.get_lags_for_frequency

    def patched_get_lags_for_frequency(freq_str, *args, **kwargs):
        if isinstance(freq_str, int):
            # LagLlamaEstimator later subtracts 1 from lag indices.
            # Returning (lag + 1) preserves the intended lag.
            return [freq_str + 1]
        return orig_fn(freq_str=freq_str, *args, **kwargs)

    ll_est.get_lags_for_frequency = patched_get_lags_for_frequency


class LagLlamaForecaster(BaseForecaster):
    def __init__(
        self,
        ckpt_path: str,
        freq: str = 'h',
        device: torch.device | None = None,
    ) -> None:
        self.name = 'lag-llama'
        self.mode = 'pretrained'

        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self._freq = freq
        self._device = device
        self._ckpt_path = ckpt_path

        torch.serialization.add_safe_globals([StudentTOutput, NegativeLogLikelihood])

        ckpt_obj = torch.load(
            self._ckpt_path,
            map_location='cpu',
            weights_only=False,
        )
        model_kwargs = ckpt_obj['hyper_parameters']['model_kwargs']
        lags_seq = [int(v) for v in model_kwargs['lags_seq']]

        _patch_lag_llama_lags_handling()

        self._estimator = LagLlamaEstimator(
            prediction_length=PREDICTION_LENGTH,
            context_length=int(model_kwargs['context_length']),
            input_size=int(model_kwargs['input_size']),
            n_layer=int(model_kwargs['n_layer']),
            n_head=int(model_kwargs['n_head']),
            n_embd_per_head=int(model_kwargs['n_embd_per_head']),
            scaling=str(model_kwargs.get('scaling', 'robust')),
            time_feat=bool(model_kwargs.get('time_feat', True)),
            dropout=float(model_kwargs.get('dropout', 0.0)),
            ckpt_path=self._ckpt_path,
            device=self._device,
            batch_size=1,
            num_parallel_samples=20,
            lags_seq=lags_seq,
            use_single_pass_sampling=True,
        )

        transformation = self._estimator.create_transformation()
        module = self._estimator.create_lightning_module(use_kv_cache=True)
        self._predictor = self._estimator.create_predictor(transformation, module)

    def forecast(self, context: np.ndarray) -> ForecastResult:
        x = np.asarray(context, dtype=np.float32)
        x = x[-MAX_CONTEXT_LENGTH:]

        start = pd.Timestamp(df['timestamp'].iloc[0])
        if len(x) != len(context):
            start = start + pd.Timedelta(hours=len(context) - len(x))

        dataset = ListDataset(
            [{'start': start, 'target': x}],
            freq=self._freq,
            one_dim_target=True,
        )
        fcst = next(iter(self._predictor.predict(dataset)))
        y_pred = np.asarray(fcst.samples, dtype=np.float32).mean(axis=0) # type: ignore
        return ForecastResult(y_pred=y_pred)


class MoiraiForecaster(BaseForecaster):
    def __init__(
        self,
        repo_id: str = 'Salesforce/moirai-1.0-R-base',
        freq: str = 'h',
        device: str = 'auto',
    ) -> None:
        self.name = 'moirai-1.0-R-base'
        self.mode = 'pretrained'

        self._freq = freq
        self._module = MoiraiModule.from_pretrained(repo_id)
        self._forecast = MoiraiForecast(
            prediction_length=PREDICTION_LENGTH,
            target_dim=1,
            feat_dynamic_real_dim=0,
            past_feat_dynamic_real_dim=0,
            context_length=min(MAX_CONTEXT_LENGTH, 168),
            module=self._module,
            num_samples=20,
        )
        self._predictor = self._forecast.create_predictor(batch_size=32, device=device)

    def forecast(self, context: np.ndarray) -> ForecastResult:
        x = np.asarray(context, dtype=np.float32)
        x = x[-MAX_CONTEXT_LENGTH:]

        start = pd.Timestamp(df['timestamp'].iloc[0])
        if len(x) != len(context):
            start = start + pd.Timedelta(hours=len(context) - len(x))

        dataset = ListDataset(
            [{'start': start, 'target': x}],
            freq=self._freq,
            one_dim_target=True,
        )
        fcst = next(iter(self._predictor.predict(dataset)))
        y_pred = np.asarray(fcst.samples, dtype=np.float32).mean(axis=0) # type: ignore
        return ForecastResult(y_pred=y_pred)


def download_lag_llama_ckpt() -> str:
    return hf_hub_download(
        repo_id='time-series-foundation-models/Lag-Llama',
        filename='lag-llama.ckpt',
    )


In [61]:
import warnings

from tqdm import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=r"Using a non-tuple sequence for multidimensional indexing.*",
)


def evaluate_forecaster(
    forecaster: BaseForecaster,
    test_years: Iterable[int],
    max_origins_per_year: int | None = None,
) -> pd.DataFrame:
    rows: list[dict[str, object]] = []

    full_series = df[["timestamp", TARGET_COL]].copy()

    for test_year in test_years:
        train_end_year = test_year - 1
        train_label = f"2013-{train_end_year}"

        start, end = year_bounds(test_year)
        test_df = full_series[
            (full_series["timestamp"] >= start) & (full_series["timestamp"] < end)
        ].reset_index(drop=True)

        seg_true: dict[str, list[np.ndarray]] = {
            "all": [],
            "pre_conflict": [],
            "post_conflict": [],
        }
        seg_pred: dict[str, list[np.ndarray]] = {
            "all": [],
            "pre_conflict": [],
            "post_conflict": [],
        }
        seg_windows: dict[str, int] = {
            "all": 0,
            "pre_conflict": 0,
            "post_conflict": 0,
        }

        origins = list(iter_test_origins(test_df))
        if max_origins_per_year is not None:
            origins = origins[:max_origins_per_year]

        for i in tqdm(origins, desc=f"{forecaster.name} {test_year}", leave=True):
            origin_ts = pd.Timestamp(test_df["timestamp"].iloc[i])

            context = full_series[full_series["timestamp"] < origin_ts][
                TARGET_COL
            ].to_numpy(dtype=np.float32)

            y_true = test_df[TARGET_COL].iloc[i : i + PREDICTION_LENGTH].to_numpy(
                dtype=np.float32
            )

            y_pred = forecaster.forecast(context).y_pred
            if len(y_pred) != len(y_true):
                raise ValueError(
                    f"Bad forecast length: got {len(y_pred)} expected {len(y_true)}"
                )

            seg = "post_conflict" if origin_ts >= CONFLICT_DATE else "pre_conflict"

            seg_true["all"].append(y_true)
            seg_pred["all"].append(y_pred)
            seg_windows["all"] += 1

            seg_true[seg].append(y_true)
            seg_pred[seg].append(y_pred)
            seg_windows[seg] += 1

        for seg in ["all", "pre_conflict", "post_conflict"]:
            if seg_windows[seg] == 0:
                continue

            y_true_all = np.concatenate(seg_true[seg])
            y_pred_all = np.concatenate(seg_pred[seg])
            m = compute_metrics(y_true_all, y_pred_all)

            rows.append(
                {
                    "model": forecaster.name,
                    "mode": forecaster.mode,
                    "train_years": train_label,
                    "test_year": test_year,
                    "segment": seg,
                    "n_windows": seg_windows[seg],
                    "n_points": int(y_true_all.size),
                    **m,
                }
            )

    return pd.DataFrame(rows)


def add_period_column(results: pd.DataFrame) -> pd.DataFrame:
    out = results.copy()
    out["period"] = np.where(out["test_year"] >= 2022, "post", "pre")
    return out


def patch_lagged_sequence_values_for_lag_llama() -> None:
    import gluonts.torch.util as gtu
    import lag_llama.model.module as ll_module
    import torch
    from gluonts.torch.util import slice_along_dim

    def stable_lagged_sequence_values(
        indices: list[int],
        prior_sequence: torch.Tensor,
        sequence: torch.Tensor,
        dim: int,
    ) -> torch.Tensor:
        seq_len = int(sequence.shape[dim])
        if seq_len <= 0:
            raise ValueError(f"sequence length must be > 0, got {seq_len}")

        full_sequence = torch.cat((prior_sequence, sequence), dim=dim)
        full_len = int(full_sequence.shape[dim])

        lags_values: list[torch.Tensor] = []
        for lag_index in indices:
            lag = int(lag_index)
            if lag < 0:
                raise ValueError(f"negative lag is not supported: {lag}")

            end = full_len - lag
            start = end - seq_len
            lags_values.append(
                slice_along_dim(full_sequence, dim=dim, slice_=slice(start, end))
            )

        return torch.stack(lags_values, dim=-1)

    gtu.lagged_sequence_values = stable_lagged_sequence_values
    ll_module.lagged_sequence_values = stable_lagged_sequence_values

In [62]:
RUN_CHRONOS = True
RUN_LAG_LLAMA = True
RUN_MOIRAI = True

SMOKE_TEST = False
TEST_YEARS = list(range(2014, 2026))
MAX_ORIGINS_PER_YEAR = 2 if SMOKE_TEST else None

In [63]:
if RUN_LAG_LLAMA:
    patch_lagged_sequence_values_for_lag_llama()

forecasters: list[BaseForecaster] = []
if RUN_CHRONOS:
    forecasters.append(Chronos2Forecaster())

if RUN_LAG_LLAMA:
    ckpt_path = download_lag_llama_ckpt()
    forecasters.append(LagLlamaForecaster(ckpt_path=ckpt_path))


if RUN_MOIRAI:
    forecasters.append(MoiraiForecaster())

In [64]:
frames: list[pd.DataFrame] = []
for forecaster in forecasters:
    try:
        frames.append(
            evaluate_forecaster(
                forecaster,
                test_years=TEST_YEARS,
                max_origins_per_year=MAX_ORIGINS_PER_YEAR,
            )
        )
    except Exception as exc:
        print(f"FAILED: {forecaster.name} ({type(exc).__name__}: {exc})")

moirai-1.0-R-base 2025: 100%|██████████| 365/365 [04:05<00:00,  1.49it/s]


In [65]:
if not frames:
    raise RuntimeError("All model runs failed; see errors above.")

all_results = pd.concat(frames, ignore_index=True)
all_results = add_period_column(all_results)

print("Per-year results (segment=all):")
print(
    all_results[all_results["segment"] == "all"]
    .sort_values(["model", "mode", "test_year"])
    .reset_index(drop=True)
)

Per-year results (segment=all):
                model        mode train_years  test_year segment  n_windows  n_points      smape           mae           mse           r2 period
0           chronos-2  pretrained   2013-2013       2014     all        365      8760   4.663107  3.275463e+04  2.548292e+09     0.978325    pre
1           chronos-2  pretrained   2013-2014       2015     all        365      8760   4.791155  3.365321e+04  2.588449e+09     0.979836    pre
2           chronos-2  pretrained   2013-2015       2016     all        366      8784  23.558141  6.271329e+06  3.379935e+15    -0.631580    pre
3           chronos-2  pretrained   2013-2016       2017     all        365      8760   4.775915  3.610642e+04  2.896299e+09     0.983594    pre
4           chronos-2  pretrained   2013-2017       2018     all        365      8760   4.398603  3.209276e+04  2.307177e+09     0.987572    pre
5           chronos-2  pretrained   2013-2018       2019     all        365      8760   5.027565  

In [66]:
print("\nConflict slice (window start vs 2022-02-24):")
conflict_table = (
    all_results[all_results["segment"].isin(["pre_conflict", "post_conflict"])]
    .groupby(["model", "mode", "period", "segment"], as_index=False)
    .agg(
        smape=("smape", "mean"),
        mae=("mae", "mean"),
        mse=("mse", "mean"),
        r2=("r2", "mean"),
        n_windows=("n_windows", "sum"),
        n_points=("n_points", "sum"),
    )
    .sort_values(["model", "mode", "period", "segment"])
)
display(conflict_table)


Conflict slice (window start vs 2022-02-24):


,model,mode,period,segment,smape,mae,mse,r2,n_windows,n_points
0,chronos-2,pretrained,post,post_conflict,6.435522,1.570417e+05,4.476653e+12,-45.344434,1407,33768
1,chronos-2,pretrained,post,pre_conflict,27.464965,2.855190e+07,1.832090e+16,-4.275922,54,1296
2,chronos-2,pretrained,pre,pre_conflict,7.930927,8.607648e+05,4.624856e+14,0.655688,2922,70128
3,lag-llama,pretrained,post,post_conflict,7.483664,4.795711e+04,5.012615e+09,0.953001,1407,33768
4,lag-llama,pretrained,post,pre_conflict,8.222929,3.742895e+06,3.486055e+15,-0.003889,54,1296
5,lag-llama,pretrained,pre,pre_conflict,7.882493,3.673361e+05,2.992516e+14,0.714850,2922,70128
6,moirai-1.0-R-base,pretrained,post,post_conflict,7.583348,4.729776e+04,4.940942e+09,0.953735,1407,33768
7,moirai-1.0-R-base,pretrained,post,pre_conflict,25.339697,4.613364e+07,8.319915e+16,-22.959096,54,1296
8,moirai-1.0-R-base,pretrained,pre,pre_conflict,9.078791,4.308189e+06,8.974495e+16,-271.952001,2922,70128


In [ ]:
print("\nSide-by-side (SMAPE) per test year:")
smape_pivot = (
    all_results[all_results["segment"] == "all"]
    .pivot_table(
        index="test_year",
        columns=["model", "mode"],
        values="smape",
        aggfunc="mean",
    )
    .sort_index()
)
display(smape_pivot)


Side-by-side (SMAPE) per test year:


model,chronos-2,lag-llama,moirai-1.0-R-base
mode,pretrained,pretrained,pretrained
test_year,,,
2014,4.663107,7.467893,7.625392
2015,4.791155,7.181727,7.384433
2016,23.558141,9.871203,14.355241
2017,4.775915,7.590860,7.553877
2018,4.398603,7.499331,7.171646
2019,5.027565,7.667586,7.711495
2020,11.700779,8.651297,14.007147
2021,4.532150,7.130047,6.821099


In [68]:
# Fine-tuning configuration (small POC)

RUN_CHRONOS_FINETUNE = True
RUN_LAG_LLAMA_FINETUNE = True
RUN_MOIRAI_FINETUNE = True

# Full expanding-window run (2014..2025).
SMOKE_TEST_FINETUNE = False
TEST_YEARS_FINETUNED = list(range(2014, 2026))

# Keep this small; full run does 12 folds (2014..2025).
FINE_TUNE_MAX_EPOCHS = 1
FINE_TUNE_LR = 5e-5

# Optional speed knob: limit origins during fine-tuned evaluation.
# Set to None to match the full evaluation protocol (can take a long time).
FINE_TUNE_MAX_ORIGINS_PER_YEAR: int | None = 50

# For training, we sample random windows from the full train history.
FINE_TUNE_BATCH_SIZE = 8
FINE_TUNE_STEPS_PER_EPOCH = 50


In [69]:
def train_slice_end(train_end_year: int) -> pd.Timestamp:
    # Training data includes all timestamps strictly before Jan 1 of (train_end_year + 1).
    return pd.Timestamp(datetime(train_end_year + 1, 1, 1))


def make_train_target(train_end_year: int) -> np.ndarray:
    end = train_slice_end(train_end_year)
    return df.loc[df["timestamp"] < end, TARGET_COL].to_numpy(dtype=np.float32)


def make_gluonts_train_dataset(train_end_year: int, freq: str = "h") -> ListDataset:
    y = make_train_target(train_end_year)
    start = pd.Timestamp(df["timestamp"].iloc[0])
    return ListDataset(
        [{"start": start, "target": y}],
        freq=freq,
        one_dim_target=True,
    )


In [70]:
from torch.utils.data import DataLoader, Dataset


class RandomWindowDataset(Dataset):
    def __init__(
        self,
        series: np.ndarray,
        context_length: int,
        prediction_length: int,
        n_samples: int,
    ) -> None:
        self.series = np.asarray(series, dtype=np.float32)
        self.context_length = int(context_length)
        self.prediction_length = int(prediction_length)
        self.n_samples = int(n_samples)

        min_total = self.context_length + self.prediction_length
        if len(self.series) < min_total:
            raise ValueError(f"Need at least {min_total} points, "
                             f"got {len(self.series)}")

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        max_start = len(self.series) - (self.context_length + self.prediction_length)
        start = int(np.random.randint(0, max_start + 1))

        past = self.series[start : start + self.context_length]
        future = self.series[
            start
            + self.context_length : start
            + self.context_length
            + self.prediction_length
        ]

        context = torch.from_numpy(past)  # (context_len,)
        future_target = torch.from_numpy(future)  # (pred_len,)

        return {
            "context": context,
            "future_target": future_target,
        }


In [71]:
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


@dataclass
class ForecastResult:
    y_pred: np.ndarray


class BaseForecaster:
    name: str
    mode: str

    def forecast(self, context: np.ndarray) -> ForecastResult:
        raise NotImplementedError


class Chronos2FineTunedForecaster(BaseForecaster):
    def __init__(
        self,
        model: "Chronos2Model",
        device: torch.device,
    ) -> None:
        self.name = "chronos-2"
        self.mode = "finetuned"
        self._model = model
        self._device = device

        qs = self._model.chronos_config.quantiles
        self._q_index = int(np.argmin(np.abs(np.asarray(qs) - 0.5)))

    def forecast(self, context: np.ndarray) -> ForecastResult:
        x = np.asarray(context, dtype=np.float32)[-MAX_CONTEXT_LENGTH:]
        ctx = torch.from_numpy(x)[None, :].to(self._device)  # (b=1, t)

        output_patch_size = int(self._model.chronos_config.output_patch_size)
        num_output_patches = int(
            (PREDICTION_LENGTH + output_patch_size - 1) // output_patch_size
        )
        with torch.no_grad():
            out = self._model(context=ctx, num_output_patches=num_output_patches)

        q = out.quantile_preds[0, self._q_index, :PREDICTION_LENGTH]
        return ForecastResult(y_pred=q.detach().cpu().numpy())


def finetune_chronos2_for_year(train_end_year: int) -> Chronos2FineTunedForecaster:
    import math

    from chronos.chronos2 import Chronos2Model

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = Chronos2Model.from_pretrained("amazon/chronos-2").to(device)

    # POC: only fine-tune the output head for speed.
    for p in model.parameters():
        p.requires_grad = False
    for p in model.output_patch_embedding.parameters():
        p.requires_grad = True

    model.train()
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=FINE_TUNE_LR,
    )

    y_train = make_train_target(train_end_year)
    context_length = int(min(MAX_CONTEXT_LENGTH, model.chronos_config.context_length))
    num_output_patches = int(
        math.ceil(PREDICTION_LENGTH / model.chronos_config.output_patch_size)
    )

    ds = RandomWindowDataset(
        series=y_train,
        context_length=context_length,
        prediction_length=PREDICTION_LENGTH,
        n_samples=FINE_TUNE_BATCH_SIZE * FINE_TUNE_STEPS_PER_EPOCH,
    )
    dl = DataLoader(ds, batch_size=FINE_TUNE_BATCH_SIZE, shuffle=False)

    total_steps = max(1, FINE_TUNE_MAX_EPOCHS * len(dl))
    pbar = tqdm(
        total=total_steps,
        desc=f"chronos-2 finetune ≤{train_end_year} ({device.type})",
        disable=not SHOW_PROGRESS,
    )

    for _epoch in range(FINE_TUNE_MAX_EPOCHS):
        for batch in dl:
            ctx = batch["context"].to(device)  # (b, t)
            fut = batch["future_target"].to(device)  # (b, pred_len)

            out = model(
                context=ctx,
                future_target=fut,
                num_output_patches=num_output_patches,
            )
            loss = out.loss
            if loss is None:
                raise RuntimeError("Chronos2Model did not return a loss.")

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            pbar.update(1)

    pbar.close()

    model.eval()
    return Chronos2FineTunedForecaster(model=model, device=device)


In [72]:
class LagLlamaFineTunedForecaster(BaseForecaster):
    def __init__(self, predictor, freq: str = "h") -> None:
        self.name = "lag-llama"
        self.mode = "finetuned"
        self._predictor = predictor
        self._freq = freq

    def forecast(self, context: np.ndarray) -> ForecastResult:
        x = np.asarray(context, dtype=np.float32)[-MAX_CONTEXT_LENGTH:]

        start = pd.Timestamp(df["timestamp"].iloc[0])
        if len(x) != len(context):
            start = start + pd.Timedelta(hours=len(context) - len(x))

        dataset = ListDataset(
            [{"start": start, "target": x}],
            freq=self._freq,
            one_dim_target=True,
        )
        fcst = next(iter(self._predictor.predict(dataset)))
        y_pred = np.asarray(fcst.samples, dtype=np.float32).mean(axis=0)  # type: ignore
        return ForecastResult(y_pred=y_pred)


def finetune_lag_llama_for_year(train_end_year: int) -> LagLlamaFineTunedForecaster:
    ckpt_path = download_lag_llama_ckpt()

    patch_lagged_sequence_values_for_lag_llama()
    _patch_lag_llama_lags_handling()

    torch.serialization.add_safe_globals([StudentTOutput, NegativeLogLikelihood])

    ckpt_obj = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model_kwargs = ckpt_obj["hyper_parameters"]["model_kwargs"]
    lags_seq = [int(v) for v in model_kwargs["lags_seq"]]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    estimator = LagLlamaEstimator(
        prediction_length=PREDICTION_LENGTH,
        context_length=int(model_kwargs["context_length"]),
        input_size=int(model_kwargs["input_size"]),
        n_layer=int(model_kwargs["n_layer"]),
        n_head=int(model_kwargs["n_head"]),
        n_embd_per_head=int(model_kwargs["n_embd_per_head"]),
        scaling=str(model_kwargs.get("scaling", "robust")),
        time_feat=bool(model_kwargs.get("time_feat", True)),
        dropout=float(model_kwargs.get("dropout", 0.0)),
        ckpt_path=ckpt_path,
        device=device,
        batch_size=32,
        num_batches_per_epoch=FINE_TUNE_STEPS_PER_EPOCH,
        lr=FINE_TUNE_LR,
        num_parallel_samples=20,
        lags_seq=lags_seq,
        use_single_pass_sampling=True,
        trainer_kwargs={
            "max_epochs": FINE_TUNE_MAX_EPOCHS,
            "logger": False,
            "enable_progress_bar": bool(SHOW_PROGRESS),
        },
    )

    train_ds = make_gluonts_train_dataset(train_end_year, freq="h")
    predictor = estimator.train(train_ds)
    return LagLlamaFineTunedForecaster(predictor=predictor, freq="h")


In [73]:
import lightning as L
from uni2ts.data.dataset import TimeSeriesDataset
from uni2ts.data.indexer._base import Indexer
from uni2ts.data.loader import DataLoader as UniDataLoader
from uni2ts.data.loader import PackCollate
from uni2ts.model.moirai.finetune import MoiraiFinetune
from uni2ts.transform.patch import DefaultPatchSizeConstraints

# Pandas >=2.2 may normalize hourly offsets to lowercase "h",
# but Uni2TS patch constraints may only define uppercase keys.
if (
    "h" not in DefaultPatchSizeConstraints.DEFAULT_RANGES
    and "H" in DefaultPatchSizeConstraints.DEFAULT_RANGES
):
    DefaultPatchSizeConstraints.DEFAULT_RANGES["h"] = DefaultPatchSizeConstraints.DEFAULT_RANGES["H"]


class _SingleSeriesIndexer(Indexer):
    def __init__(self, series: np.ndarray, freq: str = "H"):
        super().__init__(uniform=True)
        self._series = np.asarray(series, dtype=np.float32)
        self._freq = freq

    def __len__(self) -> int:
        return 1

    def _getitem_int(self, idx: int) -> dict[str, object]:
        # Uni2TS transforms expect a `freq` field compatible with pandas offsets.
        return {
            "target": self._series,
            "freq": self._freq,
            "start": pd.Timestamp(df["timestamp"].iloc[0]),
            "item_id": "series",
        }

    def _getitem_iterable(self, idx: Iterable[int]) -> dict[str, list[object]]:
        idx_list = list(idx)
        return {
            "target": [self._series for _ in idx_list],
            "freq": [self._freq for _ in idx_list],
            "start": [pd.Timestamp(df["timestamp"].iloc[0]) for _ in idx_list],
            "item_id": ["series" for _ in idx_list],
        }


class MoiraiFineTunedForecaster(BaseForecaster):
    def __init__(
        self,
        module: MoiraiModule,
        freq: str = "H",
        device: str = "auto",
    ) -> None:
        self.name = "moirai-1.0-R-base"
        self.mode = "finetuned"

        self._forecast = MoiraiForecast(
            prediction_length=PREDICTION_LENGTH,
            target_dim=1,
            feat_dynamic_real_dim=0,
            past_feat_dynamic_real_dim=0,
            context_length=min(MAX_CONTEXT_LENGTH, 168),
            module=module,
            num_samples=20,
        )
        self._predictor = self._forecast.create_predictor(batch_size=32, device=device)
        self._freq = freq

    def forecast(self, context: np.ndarray) -> ForecastResult:
        x = np.asarray(context, dtype=np.float32)[-MAX_CONTEXT_LENGTH:]

        start = pd.Timestamp(df["timestamp"].iloc[0])
        if len(x) != len(context):
            start = start + pd.Timedelta(hours=len(context) - len(x))

        dataset = ListDataset(
            [{"start": start, "target": x}],
            freq=self._freq,
            one_dim_target=True,
        )
        fcst = next(iter(self._predictor.predict(dataset)))
        y_pred = np.asarray(fcst.samples, dtype=np.float32).mean(axis=0)  # type: ignore
        return ForecastResult(y_pred=y_pred)


def finetune_moirai_for_year(train_end_year: int) -> MoiraiFineTunedForecaster:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_series = make_train_target(train_end_year)
    indexer = _SingleSeriesIndexer(train_series, freq="H")

    base_module = MoiraiModule.from_pretrained("Salesforce/moirai-1.0-R-base")

    num_training_steps = int(FINE_TUNE_MAX_EPOCHS * FINE_TUNE_STEPS_PER_EPOCH)
    num_warmup_steps = max(1, int(0.1 * num_training_steps))

    ft = MoiraiFinetune(
        min_patches=2,
        min_mask_ratio=0.1,
        max_mask_ratio=0.5,
        max_dim=1,
        num_training_steps=num_training_steps,
        num_warmup_steps=num_warmup_steps,
        module=base_module,
        lr=FINE_TUNE_LR,
        log_on_step=False,
    )

    train_transform = ft.train_transform_map["default"]()

    ds = TimeSeriesDataset(indexer=indexer, transform=train_transform, dataset_weight=100.0)

    collate = PackCollate(
        max_length=ft.module.max_seq_len,
        seq_fields=ft.seq_fields,
        pad_func_map=ft.pad_func_map,
        target_field="target",
    )

    train_loader = UniDataLoader(
        dataset=ds,
        batch_size=8,
        cycle=True,
        num_batches_per_epoch=FINE_TUNE_STEPS_PER_EPOCH,
        shuffle=False,
        num_workers=0,
        collate_fn=collate,
        pin_memory=torch.cuda.is_available(),
        drop_last=True,
        fill_last=True,
    )

    trainer = L.Trainer(
        max_epochs=FINE_TUNE_MAX_EPOCHS,
        accelerator=device,
        devices=1,
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=bool(SHOW_PROGRESS),
        log_every_n_steps=10,
    )

    trainer.fit(ft, train_dataloaders=train_loader)

    ft.module.eval()
    return MoiraiFineTunedForecaster(module=ft.module, freq="H", device="auto")


In [74]:
def evaluate_finetuned_models(test_years: list[int]) -> pd.DataFrame:
    rows: list[pd.DataFrame] = []

    for test_year in test_years:
        train_end_year = test_year - 1

        if RUN_CHRONOS_FINETUNE:
            ft = finetune_chronos2_for_year(train_end_year)
            rows.append(
                evaluate_forecaster(
                    ft,
                    test_years=[test_year],
                    max_origins_per_year=FINE_TUNE_MAX_ORIGINS_PER_YEAR,
                )
            )

        if RUN_LAG_LLAMA_FINETUNE:
            ft = finetune_lag_llama_for_year(train_end_year)
            rows.append(
                evaluate_forecaster(
                    ft,
                    test_years=[test_year],
                    max_origins_per_year=FINE_TUNE_MAX_ORIGINS_PER_YEAR,
                )
            )

        if RUN_MOIRAI_FINETUNE:
            ft = finetune_moirai_for_year(train_end_year)
            rows.append(
                evaluate_forecaster(
                    ft,
                    test_years=[test_year],
                    max_origins_per_year=FINE_TUNE_MAX_ORIGINS_PER_YEAR,
                )
            )

    return pd.concat(rows, ignore_index=True)

In [75]:
finetuned_results = evaluate_finetuned_models(TEST_YEARS_FINETUNED)
finetuned_results = add_period_column(finetuned_results)

chronos-2 2014: 100%|██████████| 50/50 [00:02<00:00, 24.03it/s]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
c:\Users\vojte\projects\school\diplomovy-projekt\.venv\Lib\site-packages\lightning\pytorch\trainer\configuration_validator.py:74: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
c:\Users\vojte\projects\school\diplomovy-projekt\.venv\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:639: Checkpoint directory c:\Users\vojte\projects\school\diplomovy-projekt\notebooks\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type               | Params
-----------------------------------------------------
0 | model         | LagLlamaModel      | 2.4 M 
1 | augmentations | ApplyAugmentations | 0     
-----------------------------------------------------
2.4 M     Trainable params
0       

Epoch 0: |          | 50/? [00:08<00:00,  5.65it/s]

Epoch 0, global step 50: 'train_loss' reached 11.35275 (best 11.35275), saving model to 'c:\\Users\\vojte\\projects\\school\\diplomovy-projekt\\notebooks\\checkpoints\\epoch=0-step=50-v5.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: |          | 50/? [00:09<00:00,  5.50it/s]


lag-llama 2014: 100%|██████████| 50/50 [00:20<00:00,  2.48it/s]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
c:\Users\vojte\projects\school\diplomovy-projekt\.venv\Lib\site-packages\lightning\pytorch\trainer\configuration_validator.py:74: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name   | Type         | Params
----------------------------------------
0 | module | MoiraiModule | 91.4 M
----------------------------------------
91.4 M    Trainable params
0         Non-trainable params
91.4 M    Total params
365.431   Total estimated model params size (MB)


...

c:\Users\vojte\projects\school\diplomovy-projekt\.venv\Lib\site-packages\lightning\pytorch\core\module.py:494: You called `self.log('train/PackedNLLLoss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`


Epoch 0: |          | 50/? [17:12<00:00,  0.05it/s, train/PackedNLLLoss=13.70]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: |          | 50/? [17:12<00:00,  0.05it/s, train/PackedNLLLoss=13.70]


moirai-1.0-R-base 2025: 100%|██████████| 50/50 [00:34<00:00,  1.45it/s]


In [76]:
print("Fine-tuned per-year results (segment=all):")
display(
    finetuned_results[finetuned_results["segment"] == "all"]
    .sort_values(["model", "mode", "test_year"])
    .reset_index(drop=True)
)

Fine-tuned per-year results (segment=all):


,model,mode,train_years,test_year,segment,n_windows,n_points,smape,mae,mse,r2,period
0,chronos-2,finetuned,2013-2013,2014,all,50,1200,3.755587,4.317320e+04,3.449579e+09,0.946282,pre
1,chronos-2,finetuned,2013-2014,2015,all,50,1200,3.705087,4.400474e+04,3.755989e+09,0.925146,pre
2,chronos-2,finetuned,2013-2015,2016,all,50,1200,3.865421,4.661907e+04,4.093506e+09,0.942452,pre
3,chronos-2,finetuned,2013-2016,2017,all,50,1200,3.322683,4.933104e+04,4.552248e+09,0.935611,pre
4,chronos-2,finetuned,2013-2017,2018,all,50,1200,3.185385,3.905807e+04,2.526320e+09,0.952617,pre
5,chronos-2,finetuned,2013-2018,2019,all,50,1200,4.022081,5.191962e+04,4.937094e+09,0.919747,pre
6,chronos-2,finetuned,2013-2019,2020,all,50,1200,3.719952,4.469642e+04,3.628820e+09,0.932288,pre
7,chronos-2,finetuned,2013-2020,2021,all,50,1200,4.508612,6.066671e+04,9.683926e+09,0.868379,pre
8,chronos-2,finetuned,2013-2021,2022,all,50,1200,7.218239,4.029031e+06,3.764834e+15,-0.004194,post
9,chronos-2,finetuned,2013-2022,2023,all,50,1200,3.450882,3.488939e+04,2.077902e+09,0.960344,post


In [77]:
print("\nFine-tuned conflict slice (window start vs 2022-02-24):")
finetuned_conflict = (
    finetuned_results[
        finetuned_results["segment"].isin(["pre_conflict", "post_conflict"])
    ]
    .groupby(["model", "mode", "period", "segment"], as_index=False)
    .agg(
        smape=("smape", "mean"),
        mae=("mae", "mean"),
        mse=("mse", "mean"),
        r2=("r2", "mean"),
        n_windows=("n_windows", "sum"),
        n_points=("n_points", "sum"),
    )
    .sort_values(["model", "mode", "period", "segment"])
)
display(finetuned_conflict)


Fine-tuned conflict slice (window start vs 2022-02-24):


,model,mode,period,segment,smape,mae,mse,r2,n_windows,n_points
0,chronos-2,finetuned,post,post_conflict,3.670910,3.871111e+04,2.778158e+09,0.948069,150,3600
1,chronos-2,finetuned,post,pre_conflict,7.218239,4.029031e+06,3.764834e+15,-0.004194,50,1200
2,chronos-2,finetuned,pre,pre_conflict,3.760601,4.743361e+04,4.578435e+09,0.927815,400,9600
3,lag-llama,finetuned,post,post_conflict,4.609255,4.914756e+04,4.147427e+09,0.922979,150,3600
4,lag-llama,finetuned,post,pre_conflict,6.884015,4.018862e+06,3.765195e+15,-0.004290,50,1200
5,lag-llama,finetuned,pre,pre_conflict,4.483190,5.714964e+04,6.794267e+09,0.893995,400,9600
6,moirai-1.0-R-base,finetuned,post,post_conflict,6.699691,7.167115e+04,1.018063e+10,0.807236,150,3600
7,moirai-1.0-R-base,finetuned,post,pre_conflict,15.386622,5.311995e+06,4.075840e+15,-0.087148,50,1200
8,moirai-1.0-R-base,finetuned,pre,pre_conflict,6.664885,8.482688e+04,1.375252e+10,0.778387,400,9600


In [78]:
print("\nFine-tuned side-by-side (SMAPE) per test year:")
finetuned_smape_pivot = (
    finetuned_results[finetuned_results["segment"] == "all"]
    .pivot_table(
        index="test_year",
        columns=["model", "mode"],
        values="smape",
        aggfunc="mean",
    )
    .sort_index()
)
display(finetuned_smape_pivot)


Fine-tuned side-by-side (SMAPE) per test year:


model,chronos-2,lag-llama,moirai-1.0-R-base
mode,finetuned,finetuned,finetuned
test_year,,,
2014,3.755587,4.141836,6.408997
2015,3.705087,4.049258,6.240944
2016,3.865421,5.244207,6.812807
2017,3.322683,4.313512,6.280590
2018,3.185385,3.235191,6.861665
2019,4.022081,4.189769,7.228529
2020,3.719952,5.279363,6.008328
2021,4.508612,5.412385,7.477216
